In [1]:
# ============================================================
# PHARMAOPT
# NOTEBOOK 09 — QUALITATIVE SUPPLY-RISK INTELLIGENCE
# ============================================================
#
# PURPOSE
#
# Transform qualitative FDA pharmaceutical supply information
# into structured, auditable operational-risk signals.
#
# PRIMARY TEXT SOURCES
#
# - FDA shortage reasons
# - FDA recall reasons
# - FDA recall distribution patterns
# - Manufacturer / company information
#
# FUTURE USE
#
# - Machine-learning features
# - Supply-risk classification
# - Decision explanations
# - Human review
#
# IMPORTANT
#
# Text-derived signals are decision-support features.
# They should not be interpreted as verified causal explanations
# without appropriate review.
# ============================================================

import pandas as pd
import numpy as np
import re
from pathlib import Path

PROJECT_DIR = Path.cwd()

print("PHARMAOPT — QUALITATIVE SUPPLY-RISK INTELLIGENCE")
print("=" * 75)
print("Project directory:", PROJECT_DIR)

PHARMAOPT — QUALITATIVE SUPPLY-RISK INTELLIGENCE
Project directory: C:\Users\jpiza\PharmaOpt


In [2]:
# ============================================================
# MYSQL CONNECTION
# ============================================================

import mysql.connector
import getpass

mysql_password = getpass.getpass(
    "Enter your MySQL password: "
)

connection = mysql.connector.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password=mysql_password,
    database="pharmaopt"
)

print("Connected to PharmaOpt MySQL database.")

Enter your MySQL password:  ········


Connected to PharmaOpt MySQL database.


In [3]:
# ============================================================
# SQL QUERY HELPER
# ============================================================

def query_df(sql, params=None):
    
    cursor = connection.cursor()
    cursor.execute(sql, params or ())
    
    rows = cursor.fetchall()
    
    columns = [
        description[0]
        for description in cursor.description
    ]
    
    cursor.close()
    
    return pd.DataFrame(
        rows,
        columns=columns
    )


print("Query helper ready.")

Query helper ready.


In [4]:
# ============================================================
# LOAD QUALITATIVE FDA SUPPLY-RISK DATA
# ============================================================

shortage_text_df = query_df(
    """
    SELECT
        shortage_id,
        generic_name,
        company_name,
        status,
        shortage_reason,
        therapeutic_category,
        dosage_form,
        linked_package_ndc,
        initial_posting_date,
        update_date
    FROM shortage_event
    """
)


recall_text_df = query_df(
    """
    SELECT
        recall_number,
        recalling_firm,
        classification,
        status,
        product_description,
        reason_for_recall,
        distribution_pattern,
        voluntary_mandated,
        recall_initiation_date,
        termination_date
    FROM recall_event
    """
)


print("PHARMAOPT — QUALITATIVE DATA LOAD")
print("=" * 75)

print(
    "Shortage records:",
    f"{len(shortage_text_df):,}"
)

print(
    "Recall records:",
    f"{len(recall_text_df):,}"
)

print(
    "\nShortage reasons available:",
    f"{shortage_text_df['shortage_reason'].notna().sum():,}"
)

print(
    "Recall reasons available:",
    f"{recall_text_df['reason_for_recall'].notna().sum():,}"
)

print(
    "Distribution patterns available:",
    f"{recall_text_df['distribution_pattern'].notna().sum():,}"
)


print("\nSAMPLE SHORTAGE TEXT")
print("=" * 75)

display(
    shortage_text_df[
        [
            "generic_name",
            "company_name",
            "shortage_reason"
        ]
    ].head(10)
)


print("\nSAMPLE RECALL TEXT")
print("=" * 75)

display(
    recall_text_df[
        [
            "recalling_firm",
            "classification",
            "reason_for_recall",
            "distribution_pattern"
        ]
    ].head(10)
)

PHARMAOPT — QUALITATIVE DATA LOAD
Shortage records: 1,601
Recall records: 17,975

Shortage reasons available: 411
Recall reasons available: 17,975
Distribution patterns available: 17,975

SAMPLE SHORTAGE TEXT


,generic_name,company_name,shortage_reason
0,Midazolam Hydrochloride Injection,"Hospira, Inc., a Pfizer Company",NaN
1,Quinapril Hydrochloride Tablet,Chartwell Molecular Holdings LLC,NaN
2,Furosemide Injection,"Avet Pharmaceuticals, Inc.",Other
3,Ropivacaine Hydrochloride Injection,"Fresenius Kabi USA, LLC",Delay in shipping of the drug
4,Clindamycin Hydrochloride Capsule,Pfizer Inc.,NaN
5,Dexmedetomidine Hydrochloride Injection,Pfizer Inc.,NaN
6,Ropivacaine Hydrochloride Injection,"Fresenius Kabi USA, LLC",NaN
7,"Methylphenidate Hydrochloride Tablet, Extended...","Vertical Pharmaceuticals, LLC",NaN
8,Frovatriptan Succinate Tablet,"Endo Pharmaceuticals, Inc.",NaN
9,Belladonna and Opium Suppository,Padagis US LLC,NaN



SAMPLE RECALL TEXT


,recalling_firm,classification,reason_for_recall,distribution_pattern
0,OurPharma LLC,Not Yet Classified,Labeling: Not Elsewhere Classified: Complaint ...,Nationwide within the U.S
1,"Cubist Pharmaceuticals, Inc.",Class I,Presence of Particulate Matter: Potential pres...,"Nationwide Foreign: Israel, Taiwan"
2,Guthy-Renker LLC,Class III,Subpotent Drug: Salicylic acid is subpotent.,U.S. Nationwide including Puerto Rico and Guam...
3,"Biocompatibles U.K., Ltd.",Class III,Defective Delivery System: Incorrect type of s...,Distributed nationwide
4,Pfizer Inc.,Class III,CGMP Deviations: Firm failed to control impuri...,Nationwide
5,Teva Pharmaceuticals USA,Class II,Failed Disintegration Specifications: Out-of-s...,Distributed nationwide in the USA
6,Pfizer Inc.,Class III,Labeling: Incorrect or Missing Lot and/or Exp ...,Nationwide in the USA
7,Strides Inc.,Class II,Failed Dissolution Specifications,nationwide
8,Alpha-Tek LLC,Class I,Marketed Without an Approved NDA/ANDA: FDA ana...,Nationwide in the US
9,Salon Technologies International Inc,Class II,GMP Deviations: FDA analysis found product to ...,FL and WA only


In [5]:
# ============================================================
# SHORTAGE REASON TEXT AUDIT
# ============================================================

shortage_reason_audit = (
    shortage_text_df["shortage_reason"]
    .fillna("MISSING")
    .astype(str)
    .str.strip()
    .value_counts()
    .reset_index()
)

shortage_reason_audit.columns = [
    "shortage_reason",
    "record_count"
]

shortage_reason_audit["percent_of_records"] = (
    shortage_reason_audit["record_count"]
    / len(shortage_text_df)
    * 100
)


print("PHARMAOPT — SHORTAGE REASON AUDIT")
print("=" * 80)

print(
    "Unique shortage-reason values:",
    f"{shortage_text_df['shortage_reason'].nunique(dropna=True):,}"
)

print("\nMOST COMMON SHORTAGE REASONS")

display(
    shortage_reason_audit.head(30)
)

PHARMAOPT — SHORTAGE REASON AUDIT
Unique shortage-reason values: 8

MOST COMMON SHORTAGE REASONS


,shortage_reason,record_count,percent_of_records
0,MISSING,1190,74.328545
1,Other,153,9.556527
2,Demand increase for the drug,105,6.558401
3,Shortage of an active ingredient,60,3.747658
4,Discontinuation of the manufacture of the drug,48,2.998126
5,Delay in shipping of the drug,21,1.311680
6,Requirements related to complying with good ma...,18,1.124297
7,Shortage of an inactive ingredient component,4,0.249844
8,Regulatory delay,2,0.124922


In [6]:
# ============================================================
# RECALL REASON PREFIX AUDIT
# ============================================================

def extract_reason_prefix(text):
    
    if pd.isna(text):
        return "MISSING"
    
    text = str(text).strip()
    
    if ":" in text:
        prefix = text.split(":", 1)[0].strip()
    else:
        prefix = text
    
    return prefix


recall_text_df["reason_prefix"] = (
    recall_text_df["reason_for_recall"]
    .apply(extract_reason_prefix)
)


recall_prefix_audit = (
    recall_text_df["reason_prefix"]
    .value_counts()
    .reset_index()
)

recall_prefix_audit.columns = [
    "reason_prefix",
    "record_count"
]

recall_prefix_audit["percent_of_recalls"] = (
    recall_prefix_audit["record_count"]
    / len(recall_text_df)
    * 100
)


print("PHARMAOPT — RECALL REASON PREFIX AUDIT")
print("=" * 80)

print(
    "Unique extracted prefixes:",
    f"{recall_text_df['reason_prefix'].nunique():,}"
)

print("\nMOST COMMON RECALL REASON PREFIXES")

display(
    recall_prefix_audit.head(40)
)

PHARMAOPT — RECALL REASON PREFIX AUDIT
Unique extracted prefixes: 1,083

MOST COMMON RECALL REASON PREFIXES


,reason_prefix,record_count,percent_of_recalls
0,Lack of Assurance of Sterility,3839,21.357441
1,CGMP Deviations,2332,12.973574
2,Labeling,1227,6.826147
3,Penicillin Cross Contamination,530,2.948540
4,Microbial contamination,465,2.586926
5,Subpotent Drug,408,2.269819
6,Lack of sterility assurance.,405,2.253129
7,Lack of Assurance of Sterility; FDA inspection...,398,2.214186
8,Failed Impurities/Degradation Specifications,395,2.197497
9,cGMP deviations,340,1.891516


In [8]:
# ============================================================
# RECALL TEXT NORMALIZATION
# ============================================================

def normalize_risk_text(text):
    
    if pd.isna(text):
        return ""
    
    text = str(text).lower()
    
    # Standardize common formatting
    text = text.replace("cgmp", "gmp")
    
    # Remove punctuation
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    
    # Collapse repeated whitespace
    text = re.sub(r"\s+", " ", text).strip()
    
    return text


recall_text_df["reason_clean"] = (
    recall_text_df["reason_for_recall"]
    .apply(normalize_risk_text)
)

recall_text_df["prefix_clean"] = (
    recall_text_df["reason_prefix"]
    .apply(normalize_risk_text)
)


clean_prefix_audit = (
    recall_text_df["prefix_clean"]
    .value_counts()
    .reset_index()
)

clean_prefix_audit.columns = [
    "normalized_prefix",
    "record_count"
]


print("PHARMAOPT — NORMALIZED RECALL LANGUAGE")
print("=" * 80)

print(
    "Original unique prefixes:",
    f"{recall_text_df['reason_prefix'].nunique():,}"
)

print(
    "Normalized unique prefixes:",
    f"{recall_text_df['prefix_clean'].nunique():,}"
)

print("\nMOST COMMON NORMALIZED PREFIXES")

display(
    clean_prefix_audit.head(40)
)

PHARMAOPT — NORMALIZED RECALL LANGUAGE
Original unique prefixes: 1,083
Normalized unique prefixes: 876

MOST COMMON NORMALIZED PREFIXES


,normalized_prefix,record_count
0,lack of assurance of sterility,4278
1,gmp deviations,3053
2,labeling,1234
3,lack of sterility assurance,688
4,penicillin cross contamination,531
5,subpotent drug,484
6,microbial contamination,465
7,failed impurities degradation specifications,457
8,marketed without an approved nda anda,430
9,lack of assurance of sterility fda inspection ...,398


In [9]:
# ============================================================
# PHARMAOPT — OPERATIONAL RECALL-RISK TAXONOMY v0.1
# ============================================================
#
# A recall may activate MULTIPLE risk categories.
#
# These categories are structured decision-support signals,
# not causal conclusions.
# ============================================================

RISK_PATTERNS = {

    "sterility_risk": (
        r"steril|non sterility|aseptic"
    ),

    "manufacturing_gmp_risk": (
        r"\bgmp\b|good manufacturing practice|"
        r"processing control|process control"
    ),

    "contamination_risk": (
        r"microbial contamination|chemical contamination|"
        r"cross contamination|bacterial contamination|"
        r"fungal contamination|mold contamination|"
        r"penicillin cross contamination"
    ),

    "particulate_foreign_matter_risk": (
        r"particulate|foreign substance|foreign matter|"
        r"foreign tablet|foreign capsule"
    ),

    "potency_dosage_risk": (
        r"subpotent|superpotent|potency|"
        r"content uniformity"
    ),

    "specification_failure_risk": (
        r"failed .* specification|"
        r"failed .* specifications|"
        r"out of specification|"
        r"dissolution specification|"
        r"disintegration specification|"
        r"impurities degradation specification"
    ),

    "stability_expiry_risk": (
        r"stability|expiry|expiration"
    ),

    "packaging_delivery_risk": (
        r"defective container|defective delivery|"
        r"delivery system|packaging|package defect|"
        r"container closure"
    ),

    "labeling_risk": (
        r"\blabeling\b|\blabel\b|mislabel"
    ),

    "regulatory_approval_risk": (
        r"without an approved nda|"
        r"without an approved anda|"
        r"approved nda anda|"
        r"unapproved drug|"
        r"marketed without"
    ),

    "storage_temperature_risk": (
        r"temperature abuse|temperature excursion|"
        r"storage condition|improper storage"
    )
}


for category, pattern in RISK_PATTERNS.items():

    recall_text_df[category] = (
        recall_text_df["reason_clean"]
        .str.contains(
            pattern,
            regex=True,
            na=False
        )
        .astype(int)
    )


risk_columns = list(RISK_PATTERNS.keys())


# Number of different risk signals activated by each recall
recall_text_df["risk_signal_count"] = (
    recall_text_df[risk_columns]
    .sum(axis=1)
)


# Flag recalls not yet captured by the taxonomy
recall_text_df["unclassified_risk"] = (
    recall_text_df["risk_signal_count"] == 0
).astype(int)


print("PHARMAOPT — OPERATIONAL RISK TAXONOMY v0.1")
print("=" * 80)

print(
    "Recall records classified:",
    f"{(recall_text_df['unclassified_risk'] == 0).sum():,}"
)

print(
    "Recall records unclassified:",
    f"{recall_text_df['unclassified_risk'].sum():,}"
)

print(
    "Taxonomy coverage:",
    f"{(1 - recall_text_df['unclassified_risk'].mean()):.2%}"
)

PHARMAOPT — OPERATIONAL RISK TAXONOMY v0.1
Recall records classified: 17,343
Recall records unclassified: 632
Taxonomy coverage: 96.48%


In [10]:
# ============================================================
# TAXONOMY COVERAGE AUDIT
# ============================================================

risk_summary = pd.DataFrame({

    "risk_category": risk_columns,

    "recall_count": [
        recall_text_df[column].sum()
        for column in risk_columns
    ]

})


risk_summary["percent_of_recalls"] = (
    risk_summary["recall_count"]
    / len(recall_text_df)
    * 100
)


risk_summary = (
    risk_summary
    .sort_values(
        "recall_count",
        ascending=False
    )
    .reset_index(drop=True)
)


print("PHARMAOPT — TAXONOMY COVERAGE AUDIT")
print("=" * 80)

display(
    risk_summary.round(2)
)


print("\nRISK SIGNALS PER RECALL")
print("=" * 80)

display(
    recall_text_df["risk_signal_count"]
    .value_counts()
    .sort_index()
    .rename_axis("risk_signal_count")
    .reset_index(name="recall_count")
)


print("\nSAMPLE UNCLASSIFIED RECALL REASONS")
print("=" * 80)

display(
    recall_text_df.loc[
        recall_text_df["unclassified_risk"] == 1,
        [
            "classification",
            "reason_for_recall"
        ]
    ]
    .drop_duplicates()
    .head(30)
)

PHARMAOPT — TAXONOMY COVERAGE AUDIT


,risk_category,recall_count,percent_of_recalls
0,sterility_risk,6709,37.32
1,manufacturing_gmp_risk,4338,24.13
2,contamination_risk,1919,10.68
3,specification_failure_risk,1796,9.99
4,stability_expiry_risk,1653,9.20
5,labeling_risk,1568,8.72
6,potency_dosage_risk,1010,5.62
7,particulate_foreign_matter_risk,882,4.91
8,regulatory_approval_risk,584,3.25
9,packaging_delivery_risk,467,2.60



RISK SIGNALS PER RECALL


,risk_signal_count,recall_count
0,0,632
1,1,13527
2,2,3589
3,3,224
4,4,3



SAMPLE UNCLASSIFIED RECALL REASONS


,classification,reason_for_recall
32,Class I,Product Mix-up: Incorrect diluent component in...
124,Class II,Discoloration: Presence of dark discoloration ...
130,Class II,Crystallization
154,Class II,Incorrect/Undeclared Excipients: The inactive ...
170,Class II,Correct Labeled Product Mispack: confirmed rep...
178,Class II,Crystallization: Formation of crystals observe...
189,Class II,Discoloration
226,Class II,Resuspension Problems: Recalled lot did not me...
341,Class II,Product mix up: complaint received that seale...
342,Class II,Crystallization: Presence of crystals of Nimod...


In [11]:
# Show the two results needed for the next step

print("TAXONOMY COVERAGE AUDIT")
print("=" * 80)
print(risk_summary.round(2).to_string(index=False))

print("\nRISK SIGNALS PER RECALL")
print("=" * 80)

signal_counts = (
    recall_text_df["risk_signal_count"]
    .value_counts()
    .sort_index()
    .rename_axis("risk_signal_count")
    .reset_index(name="recall_count")
)

print(signal_counts.to_string(index=False))

TAXONOMY COVERAGE AUDIT
                  risk_category  recall_count  percent_of_recalls
                 sterility_risk          6709               37.32
         manufacturing_gmp_risk          4338               24.13
             contamination_risk          1919               10.68
     specification_failure_risk          1796                9.99
          stability_expiry_risk          1653                9.20
                  labeling_risk          1568                8.72
            potency_dosage_risk          1010                5.62
particulate_foreign_matter_risk           882                4.91
       regulatory_approval_risk           584                3.25
        packaging_delivery_risk           467                2.60
       storage_temperature_risk           463                2.58

RISK SIGNALS PER RECALL
 risk_signal_count  recall_count
                 0           632
                 1         13527
                 2          3589
                 3         

In [12]:
# ============================================================
# PHARMAOPT — OPERATIONAL RECALL-RISK TAXONOMY v0.2
# ============================================================
#
# v0.2 expands the original taxonomy using failure modes
# observed during the v0.1 unclassified-record audit.
#
# New concepts include:
#
# - Product mix-ups / formulation errors
# - Physical product-quality failures
# - Additional spelling / terminology variants
#
# Multi-label classification is intentionally preserved.
# ============================================================

RISK_PATTERNS_V2 = {

    "sterility_risk": (
        r"steril|non sterility|aseptic"
    ),

    "manufacturing_gmp_risk": (
        r"\bgmps?\b|good manufacturing practice|"
        r"processing control|process control"
    ),

    "contamination_risk": (
        r"microbial contamination|chemical contamination|"
        r"cross contamination|bacterial contamination|"
        r"fungal contamination|mold contamination|"
        r"penicillin cross contamination"
    ),

    "particulate_foreign_matter_risk": (
        r"particulate|particular matter|foreign substance|"
        r"foreign matter|foreign object|"
        r"foreign tablet|foreign capsule"
    ),

    "potency_dosage_risk": (
        r"sub[\s-]*potent|super[\s-]*potent|potency|"
        r"content uniformity"
    ),

    "specification_failure_risk": (
        r"failed .* specification|"
        r"failed .* specifications|"
        r"out of specification|"
        r"\boos\b|"
        r"dissolution specification|"
        r"disintegration specification|"
        r"impurity degradation|"
        r"impurities degradation"
    ),

    "stability_expiry_risk": (
        r"stability|expiry|expiration"
    ),

    "packaging_delivery_risk": (
        r"defective container|defective delivery|"
        r"delivery system|packaging|package defect|"
        r"container closure|mispack"
    ),

    "labeling_risk": (
        r"\blabeling\b|\blabel\b|mislabel|"
        r"wrong id|imprinted with wrong"
    ),

    "regulatory_approval_risk": (
        r"without an approved nda|"
        r"without an approved anda|"
        r"approved nda anda|"
        r"unapproved drug|"
        r"marketed without"
    ),

    "storage_temperature_risk": (
        r"temperature abuse|temperature excursion|"
        r"storage condition|improper storage"
    ),

    # --------------------------------------------------------
    # NEW IN v0.2
    # --------------------------------------------------------

    "product_identity_formulation_risk": (
        r"product mix[\s-]*up|"
        r"incorrect product formulation|"
        r"incorrect formulation|"
        r"incorrect.*ingredient|"
        r"incorrect.*diluent|"
        r"undeclared excipient|"
        r"incorrect.*excipient|"
        r"wrong product"
    ),

    "physical_product_quality_risk": (
        r"discoloration|"
        r"crystallization|"
        r"tablet separation|"
        r"resuspension|"
        r"moisture limit|"
        r"viscosity|"
        r"underfilled|"
        r"underfill"
    )
}


# Apply v0.2 taxonomy
for category, pattern in RISK_PATTERNS_V2.items():

    recall_text_df[category] = (
        recall_text_df["reason_clean"]
        .str.contains(
            pattern,
            regex=True,
            na=False
        )
        .astype(int)
    )


risk_columns_v2 = list(RISK_PATTERNS_V2.keys())


recall_text_df["risk_signal_count_v2"] = (
    recall_text_df[risk_columns_v2]
    .sum(axis=1)
)


recall_text_df["unclassified_risk_v2"] = (
    recall_text_df["risk_signal_count_v2"] == 0
).astype(int)


classified_v2 = (
    recall_text_df["unclassified_risk_v2"] == 0
).sum()

unclassified_v2 = (
    recall_text_df["unclassified_risk_v2"].sum()
)

coverage_v2 = (
    1 - recall_text_df["unclassified_risk_v2"].mean()
)


print("PHARMAOPT — OPERATIONAL RISK TAXONOMY v0.2")
print("=" * 80)

print(
    "Recall records classified:",
    f"{classified_v2:,}"
)

print(
    "Recall records unclassified:",
    f"{unclassified_v2:,}"
)

print(
    "Taxonomy coverage:",
    f"{coverage_v2:.2%}"
)

print(
    "\nCoverage improvement vs v0.1:",
    f"{coverage_v2 - (1 - recall_text_df['unclassified_risk'].mean()):.2%}"
)

PHARMAOPT — OPERATIONAL RISK TAXONOMY v0.2
Recall records classified: 17,545
Recall records unclassified: 430
Taxonomy coverage: 97.61%

Coverage improvement vs v0.1: 1.12%


In [13]:
# ============================================================
# PHARMAOPT — TAXONOMY v0.2 VALIDATION
# ============================================================

risk_summary_v2 = pd.DataFrame({

    "risk_category": risk_columns_v2,

    "recall_count": [
        recall_text_df[column].sum()
        for column in risk_columns_v2
    ]

})


risk_summary_v2["percent_of_recalls"] = (
    risk_summary_v2["recall_count"]
    / len(recall_text_df)
    * 100
)


risk_summary_v2 = (
    risk_summary_v2
    .sort_values(
        "recall_count",
        ascending=False
    )
    .reset_index(drop=True)
)


signal_counts_v2 = (
    recall_text_df["risk_signal_count_v2"]
    .value_counts()
    .sort_index()
    .rename_axis("risk_signal_count")
    .reset_index(name="recall_count")
)


remaining_unclassified = (
    recall_text_df.loc[
        recall_text_df["unclassified_risk_v2"] == 1,
        [
            "reason_prefix",
            "reason_for_recall"
        ]
    ]
    .drop_duplicates()
)


print("TAXONOMY v0.2 CATEGORY COUNTS")
print("=" * 80)
print(
    risk_summary_v2
    .round(2)
    .to_string(index=False)
)


print("\nRISK SIGNALS PER RECALL")
print("=" * 80)
print(
    signal_counts_v2
    .to_string(index=False)
)


print("\nREMAINING UNCLASSIFIED RECORDS")
print("=" * 80)

print(
    f"{unclassified_v2:,} of {len(recall_text_df):,}"
)


print("\nMOST COMMON REMAINING UNCLASSIFIED PREFIXES")
print("=" * 80)

unclassified_prefix_counts = (
    recall_text_df.loc[
        recall_text_df["unclassified_risk_v2"] == 1,
        "prefix_clean"
    ]
    .value_counts()
    .head(30)
)

print(
    unclassified_prefix_counts
    .to_string()
)

TAXONOMY v0.2 CATEGORY COUNTS
                    risk_category  recall_count  percent_of_recalls
                   sterility_risk          6709               37.32
           manufacturing_gmp_risk          4342               24.16
               contamination_risk          1919               10.68
       specification_failure_risk          1830               10.18
            stability_expiry_risk          1653                9.20
                    labeling_risk          1577                8.77
              potency_dosage_risk          1045                5.81
  particulate_foreign_matter_risk           884                4.92
         regulatory_approval_risk           584                3.25
          packaging_delivery_risk           481                2.68
         storage_temperature_risk           463                2.58
    physical_product_quality_risk           167                0.93
product_identity_formulation_risk           139                0.77

RISK SIGNALS PER 

In [14]:
# ============================================================
# PHARMAOPT — RESIDUAL NLP AUDIT
# ============================================================
#
# PURPOSE
#
# Analyze recall descriptions not captured by the interpretable
# v0.2 taxonomy.
#
# Rather than forcing 100% rule-based coverage, PharmaOpt
# preserves these residual records for additional NLP analysis.
# ============================================================

residual_df = (
    recall_text_df.loc[
        recall_text_df["unclassified_risk_v2"] == 1
    ]
    .copy()
)


# Count exact normalized residual descriptions
residual_reason_counts = (
    residual_df["reason_clean"]
    .value_counts()
    .reset_index()
)

residual_reason_counts.columns = [
    "normalized_reason",
    "record_count"
]


# Calculate concentration
largest_residual_count = (
    residual_reason_counts["record_count"].iloc[0]
)

largest_residual_share = (
    largest_residual_count
    / len(residual_df)
)


unique_residual_reasons = (
    residual_df["reason_clean"].nunique()
)


print("PHARMAOPT — RESIDUAL NLP AUDIT")
print("=" * 80)

print(
    "Total residual recalls:",
    f"{len(residual_df):,}"
)

print(
    "Unique residual descriptions:",
    f"{unique_residual_reasons:,}"
)

print(
    "Largest repeated residual:",
    f"{largest_residual_count:,}"
)

print(
    "Share represented by largest residual:",
    f"{largest_residual_share:.2%}"
)


print("\nTOP RESIDUAL DESCRIPTIONS")
print("=" * 80)

display(
    residual_reason_counts.head(20)
)

PHARMAOPT — RESIDUAL NLP AUDIT
Total residual recalls: 430
Unique residual descriptions: 72
Largest repeated residual: 298
Share represented by largest residual: 69.30%

TOP RESIDUAL DESCRIPTIONS


,normalized_reason,record_count
0,the firm received seven reports of adverse rea...,298
1,gcmp deviations fda analysis confirmed presenc...,18
2,this recall is being initiated following fda s...,15
3,ggmp deviations possible contamination with im...,6
4,presence of precipitate white substance confir...,5
5,does not meet usp or otc monograph product did...,4
6,an extraneous peak was observed for dissolutio...,3
7,short fill,3
8,an error occurred where the product was manufa...,3
9,some bottles may contain mixed strengths of th...,3


In [15]:
# ============================================================
# PHARMAOPT — RESIDUAL NLP DATASET
# ============================================================
#
# Each unique residual reason is treated as one NLP observation.
#
# record_count is preserved separately so we can later measure
# how frequently each discovered text pattern occurs.
# ============================================================

residual_nlp_df = (
    residual_reason_counts[
        residual_reason_counts["normalized_reason"].str.len() > 0
    ]
    .copy()
    .reset_index(drop=True)
)


print("PHARMAOPT — RESIDUAL NLP DATASET")
print("=" * 80)

print(
    "Unique residual descriptions:",
    f"{len(residual_nlp_df):,}"
)

print(
    "Underlying recall records represented:",
    f"{residual_nlp_df['record_count'].sum():,}"
)

print(
    "Average records per unique description:",
    f"{residual_nlp_df['record_count'].mean():.2f}"
)

display(
    residual_nlp_df.head(10)
)

PHARMAOPT — RESIDUAL NLP DATASET
Unique residual descriptions: 72
Underlying recall records represented: 430
Average records per unique description: 5.97


,normalized_reason,record_count
0,the firm received seven reports of adverse rea...,298
1,gcmp deviations fda analysis confirmed presenc...,18
2,this recall is being initiated following fda s...,15
3,ggmp deviations possible contamination with im...,6
4,presence of precipitate white substance confir...,5
5,does not meet usp or otc monograph product did...,4
6,an extraneous peak was observed for dissolutio...,3
7,short fill,3
8,an error occurred where the product was manufa...,3
9,some bottles may contain mixed strengths of th...,3


In [16]:
# ============================================================
# PHARMAOPT — TF-IDF TEXT REPRESENTATION
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer


tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=1,
    max_features=1000,
    sublinear_tf=True
)


tfidf_matrix = tfidf_vectorizer.fit_transform(
    residual_nlp_df["normalized_reason"]
)


feature_names = (
    tfidf_vectorizer.get_feature_names_out()
)


print("PHARMAOPT — TF-IDF REPRESENTATION")
print("=" * 80)

print(
    "Residual descriptions:",
    f"{tfidf_matrix.shape[0]:,}"
)

print(
    "NLP text features:",
    f"{tfidf_matrix.shape[1]:,}"
)

print(
    "Matrix shape:",
    tfidf_matrix.shape
)

PHARMAOPT — TF-IDF REPRESENTATION
Residual descriptions: 72
NLP text features: 916
Matrix shape: (72, 916)


In [17]:
# ============================================================
# PHARMAOPT — NLP CLUSTER SELECTION
# ============================================================

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


cluster_results = []


for k in range(3, 11):

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = model.fit_predict(
        tfidf_matrix
    )

    score = silhouette_score(
        tfidf_matrix,
        labels,
        metric="cosine"
    )

    cluster_results.append({
        "n_clusters": k,
        "silhouette_score": score
    })


cluster_evaluation_df = pd.DataFrame(
    cluster_results
)


best_k = int(
    cluster_evaluation_df.loc[
        cluster_evaluation_df[
            "silhouette_score"
        ].idxmax(),
        "n_clusters"
    ]
)


print("PHARMAOPT — NLP CLUSTER SELECTION")
print("=" * 80)

print(
    cluster_evaluation_df
    .round(4)
    .to_string(index=False)
)

print("\nBest-performing cluster count:", best_k)

PHARMAOPT — NLP CLUSTER SELECTION
 n_clusters  silhouette_score
          3            0.0363
          4            0.0432
          5            0.0527
          6            0.0521
          7            0.0532
          8            0.0634
          9            0.0696
         10            0.0747

Best-performing cluster count: 10


In [18]:
# ============================================================
# PHARMAOPT — RESIDUAL NLP CLUSTER MODEL
# ============================================================
#
# The best tested silhouette score occurred at k = 10.
#
# IMPORTANT:
# The absolute silhouette score is low, so these clusters are
# exploratory and should NOT yet be interpreted as validated
# operational-risk categories.
# ============================================================

final_cluster_model = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=20
)

residual_nlp_df["cluster"] = (
    final_cluster_model.fit_predict(tfidf_matrix)
)


# ------------------------------------------------------------
# CLUSTER SIZE SUMMARY
# ------------------------------------------------------------

cluster_summary = (
    residual_nlp_df
    .groupby("cluster")
    .agg(
        unique_descriptions=("normalized_reason", "count"),
        underlying_recalls=("record_count", "sum")
    )
    .reset_index()
)


# ------------------------------------------------------------
# EXTRACT TOP TF-IDF TERMS FOR EACH CLUSTER
# ------------------------------------------------------------

top_terms_per_cluster = []

ordered_centroids = (
    final_cluster_model.cluster_centers_
    .argsort(axis=1)[:, ::-1]
)


for cluster_id in range(best_k):

    top_indices = ordered_centroids[
        cluster_id,
        :10
    ]

    top_terms = [
        feature_names[index]
        for index in top_indices
    ]

    top_terms_per_cluster.append({
        "cluster": cluster_id,
        "top_terms": ", ".join(top_terms)
    })


cluster_terms_df = pd.DataFrame(
    top_terms_per_cluster
)


cluster_summary = cluster_summary.merge(
    cluster_terms_df,
    on="cluster",
    how="left"
)


print("PHARMAOPT — RESIDUAL NLP CLUSTERS")
print("=" * 100)

print(
    cluster_summary
    .to_string(index=False)
)

PHARMAOPT — RESIDUAL NLP CLUSTERS
 cluster  unique_descriptions  underlying_recalls                                                                                                                                              top_terms
       0                   13                 315                     failed, dissolution, misbranded, testing, dissolution testing, test, reconstitution, misprint, misprint tablet, failed dissolution
       1                    8                  12                                       meet, monograph, does, does meet, specification, usp, meet monograph, monograph product, meet usp, otc monograph
       2                   10                  14                                                             short, bottles, count, contain, labeled, tablets, contained, mg, count bottles, mg tablets
       3                    3                  39 deviations, impurity, fda, contamination, ggmp, deviations possible, contamination impurity, nitrosodimethylamin

In [19]:
# ============================================================
# PHARMAOPT — CLUSTER INTERPRETATION AUDIT
# ============================================================

print("PHARMAOPT — NLP CLUSTER INTERPRETATION")
print("=" * 100)


for cluster_id in range(best_k):

    cluster_data = (
        residual_nlp_df.loc[
            residual_nlp_df["cluster"] == cluster_id,
            [
                "normalized_reason",
                "record_count"
            ]
        ]
        .sort_values(
            "record_count",
            ascending=False
        )
    )

    print("\n" + "=" * 100)
    print(f"CLUSTER {cluster_id}")
    print("=" * 100)

    print(
        "Unique descriptions:",
        len(cluster_data)
    )

    print(
        "Underlying recalls:",
        cluster_data["record_count"].sum()
    )

    print("\nTOP DESCRIPTIONS:")

    for row in cluster_data.head(5).itertuples():

        description = row.normalized_reason

        # Shorten only for easier notebook display
        if len(description) > 220:
            description = description[:220] + "..."

        print(
            f"\n[{row.record_count} recall(s)] "
            f"{description}"
        )

PHARMAOPT — NLP CLUSTER INTERPRETATION

CLUSTER 0
Unique descriptions: 13
Underlying recalls: 315

TOP DESCRIPTIONS:

[298 recall(s)] the firm received seven reports of adverse reactions in the form of skin abscesses potentially linked to compounded preservative free methylprednisolone 80mg ml 10 ml vials

[3 recall(s)] an extraneous peak was observed for dissolution testing

[3 recall(s)] products failed the antimicrobial effectiveness test

[2 recall(s)] misprint on tablet

[1 recall(s)] staq pharma inc received consumer complaints related to leaking bags and other potential bag concerns upon investigation they determine the source of the issue was with the bag manufacturer who released bags lot 134142 0...

CLUSTER 1
Unique descriptions: 8
Underlying recalls: 12

TOP DESCRIPTIONS:

[4 recall(s)] does not meet usp or otc monograph product did not meet the over the counter otc monograph m023

[2 recall(s)] does not meet monograph budesonide may be slightly above or below the specifica

In [20]:
# ============================================================
# PHARMAOPT — WEAKLY SUPERVISED NLP DATASET
# ============================================================
#
# The interpretable v0.2 taxonomy provides weak labels.
#
# We train only on recalls classified by the taxonomy.
# Unclassified recalls are preserved for later inference.
#
# IMPORTANT:
# These are weak labels generated from auditable rules,
# NOT manually verified ground-truth labels.
# ============================================================

nlp_labeled_df = (
    recall_text_df.loc[
        recall_text_df["unclassified_risk_v2"] == 0
    ]
    .copy()
    .reset_index(drop=True)
)


nlp_residual_df = (
    recall_text_df.loc[
        recall_text_df["unclassified_risk_v2"] == 1
    ]
    .copy()
    .reset_index(drop=True)
)


X_text = nlp_labeled_df["reason_clean"]

Y_labels = nlp_labeled_df[risk_columns_v2]


print("PHARMAOPT — WEAKLY SUPERVISED NLP DATASET")
print("=" * 80)

print(
    "Labeled recall records:",
    f"{len(nlp_labeled_df):,}"
)

print(
    "Residual records reserved for inference:",
    f"{len(nlp_residual_df):,}"
)

print(
    "Risk categories:",
    len(risk_columns_v2)
)

print(
    "Total positive category labels:",
    f"{Y_labels.sum().sum():,}"
)

print(
    "\nAverage labels per classified recall:",
    f"{Y_labels.sum(axis=1).mean():.3f}"
)

PHARMAOPT — WEAKLY SUPERVISED NLP DATASET
Labeled recall records: 17,545
Residual records reserved for inference: 430
Risk categories: 13
Total positive category labels: 21,793

Average labels per classified recall: 1.242


In [21]:
# ============================================================
# PHARMAOPT — NLP TRAIN / TEST SPLIT
# ============================================================

from sklearn.model_selection import train_test_split


X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text,
    Y_labels,
    test_size=0.20,
    random_state=42
)


print("PHARMAOPT — NLP TRAIN / TEST SPLIT")
print("=" * 80)

print(
    "Training records:",
    f"{len(X_train_text):,}"
)

print(
    "Holdout test records:",
    f"{len(X_test_text):,}"
)

print(
    "Training share:",
    f"{len(X_train_text) / len(X_text):.2%}"
)

print(
    "Testing share:",
    f"{len(X_test_text) / len(X_text):.2%}"
)

PHARMAOPT — NLP TRAIN / TEST SPLIT
Training records: 14,036
Holdout test records: 3,509
Training share: 80.00%
Testing share: 20.00%


In [25]:

# ============================================================
# PHARMAOPT — HYBRID WORD + CHARACTER TF-IDF
# ============================================================

from sklearn.pipeline import FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer


word_tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_features=15000,
    sublinear_tf=True
)


character_tfidf = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_features=15000,
    sublinear_tf=True
)


hybrid_vectorizer = FeatureUnion([
    ("word_tfidf", word_tfidf),
    ("character_tfidf", character_tfidf)
])


X_train_nlp = hybrid_vectorizer.fit_transform(
    X_train_text
)

X_test_nlp = hybrid_vectorizer.transform(
    X_test_text
)


print("PHARMAOPT — HYBRID NLP REPRESENTATION")
print("=" * 80)

print(
    "Training matrix:",
    X_train_nlp.shape
)

print(
    "Holdout matrix:",
    X_test_nlp.shape
)

print(
    "Total NLP features:",
    f"{X_train_nlp.shape[1]:,}"
)

PHARMAOPT — HYBRID NLP REPRESENTATION
Training matrix: (14036, 25692)
Holdout matrix: (3509, 25692)
Total NLP features: 25,692


In [26]:
# ============================================================
# PHARMAOPT — LEAKAGE-RESISTANT NLP HOLDOUT
# ============================================================
#
# Exact normalized recall descriptions are kept entirely in
# either training OR testing.
#
# This prevents repeated FDA descriptions from artificially
# inflating holdout performance.
# ============================================================

from sklearn.model_selection import GroupShuffleSplit


groups = nlp_labeled_df["reason_clean"]


group_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)


train_idx, test_idx = next(
    group_split.split(
        nlp_labeled_df,
        groups=groups
    )
)


X_train_text = (
    nlp_labeled_df
    .iloc[train_idx]["reason_clean"]
)

X_test_text = (
    nlp_labeled_df
    .iloc[test_idx]["reason_clean"]
)


y_train = (
    nlp_labeled_df
    .iloc[train_idx][risk_columns_v2]
)

y_test = (
    nlp_labeled_df
    .iloc[test_idx][risk_columns_v2]
)


train_unique_text = set(X_train_text)
test_unique_text = set(X_test_text)

text_overlap = (
    train_unique_text
    .intersection(test_unique_text)
)


print("PHARMAOPT — LEAKAGE-RESISTANT NLP HOLDOUT")
print("=" * 80)

print(
    "Training records:",
    f"{len(X_train_text):,}"
)

print(
    "Holdout records:",
    f"{len(X_test_text):,}"
)

print(
    "Unique training descriptions:",
    f"{X_train_text.nunique():,}"
)

print(
    "Unique holdout descriptions:",
    f"{X_test_text.nunique():,}"
)

print(
    "Exact text overlap:",
    len(text_overlap)
)


if len(text_overlap) == 0:
    print("\nVALIDATION PASSED — no exact recall text leakage.")
else:
    print("\nWARNING — text overlap detected.")

PHARMAOPT — LEAKAGE-RESISTANT NLP HOLDOUT
Training records: 12,363
Holdout records: 5,182
Unique training descriptions: 3,304
Unique holdout descriptions: 827
Exact text overlap: 0

VALIDATION PASSED — no exact recall text leakage.


In [27]:
# ============================================================
# PHARMAOPT — LEAKAGE-RESISTANT HYBRID NLP FEATURES
# ============================================================

hybrid_vectorizer = FeatureUnion([

    (
        "word_tfidf",
        TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 2),
            min_df=2,
            max_features=15000,
            sublinear_tf=True
        )
    ),

    (
        "character_tfidf",
        TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            max_features=15000,
            sublinear_tf=True
        )
    )
])


X_train_nlp = (
    hybrid_vectorizer
    .fit_transform(X_train_text)
)

X_test_nlp = (
    hybrid_vectorizer
    .transform(X_test_text)
)


print("PHARMAOPT — FINAL HYBRID NLP REPRESENTATION")
print("=" * 80)

print(
    "Training matrix:",
    X_train_nlp.shape
)

print(
    "Holdout matrix:",
    X_test_nlp.shape
)

print(
    "Total NLP features:",
    f"{X_train_nlp.shape[1]:,}"
)

PHARMAOPT — FINAL HYBRID NLP REPRESENTATION
Training matrix: (12363, 25357)
Holdout matrix: (5182, 25357)
Total NLP features: 25,357


In [28]:
# ============================================================
# PHARMAOPT — MULTI-LABEL NLP CLASSIFIER v0.1
# ============================================================

from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression


nlp_classifier = OneVsRestClassifier(

    LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        solver="liblinear",
        random_state=42
    )

)


print("Training PharmaOpt NLP classifier...")


nlp_classifier.fit(
    X_train_nlp,
    y_train
)


print("\nTraining complete.")
print(
    "Risk categories modeled:",
    len(risk_columns_v2)
)

Training PharmaOpt NLP classifier...

Training complete.
Risk categories modeled: 13


In [29]:
# ============================================================
# PHARMAOPT — NLP HOLDOUT EVALUATION
# ============================================================

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    hamming_loss
)


y_pred = nlp_classifier.predict(
    X_test_nlp
)


micro_precision = precision_score(
    y_test,
    y_pred,
    average="micro",
    zero_division=0
)

micro_recall = recall_score(
    y_test,
    y_pred,
    average="micro",
    zero_division=0
)

micro_f1 = f1_score(
    y_test,
    y_pred,
    average="micro",
    zero_division=0
)

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

model_hamming_loss = hamming_loss(
    y_test,
    y_pred
)


print("PHARMAOPT — NLP HOLDOUT PERFORMANCE")
print("=" * 80)

print(
    "Micro Precision:",
    f"{micro_precision:.4f}"
)

print(
    "Micro Recall:",
    f"{micro_recall:.4f}"
)

print(
    "Micro F1:",
    f"{micro_f1:.4f}"
)

print(
    "Macro F1:",
    f"{macro_f1:.4f}"
)

print(
    "Hamming Loss:",
    f"{model_hamming_loss:.4f}"
)


# ------------------------------------------------------------
# CATEGORY-LEVEL PERFORMANCE
# ------------------------------------------------------------

category_results = []


for i, category in enumerate(risk_columns_v2):

    category_results.append({

        "risk_category": category,

        "test_positives": int(
            y_test.iloc[:, i].sum()
        ),

        "precision": precision_score(
            y_test.iloc[:, i],
            y_pred[:, i],
            zero_division=0
        ),

        "recall": recall_score(
            y_test.iloc[:, i],
            y_pred[:, i],
            zero_division=0
        ),

        "f1": f1_score(
            y_test.iloc[:, i],
            y_pred[:, i],
            zero_division=0
        )

    })


category_performance_df = pd.DataFrame(
    category_results
)


print("\nCATEGORY-LEVEL PERFORMANCE")
print("=" * 80)

print(
    category_performance_df
    .round(4)
    .to_string(index=False)
)

PHARMAOPT — NLP HOLDOUT PERFORMANCE
Micro Precision: 0.9939
Micro Recall: 0.9755
Micro F1: 0.9846
Macro F1: 0.9432
Hamming Loss: 0.0027

CATEGORY-LEVEL PERFORMANCE
                    risk_category  test_positives  precision  recall     f1
                   sterility_risk            2773     1.0000  0.9719 0.9857
           manufacturing_gmp_risk            1137     1.0000  1.0000 1.0000
               contamination_risk             225     0.9713  0.9022 0.9355
  particulate_foreign_matter_risk             247     1.0000  1.0000 1.0000
              potency_dosage_risk             138     1.0000  1.0000 1.0000
       specification_failure_risk             496     0.9798  0.9798 0.9798
            stability_expiry_risk             254     0.9960  0.9921 0.9941
          packaging_delivery_risk             116     1.0000  0.9569 0.9780
                    labeling_risk             266     0.9882  0.9436 0.9654
         regulatory_approval_risk             149     1.0000  1.0000 1.0000


In [30]:
# ============================================================
# PHARMAOPT — MODEL-ASSISTED RESIDUAL CLASSIFICATION
# ============================================================
#
# Apply the trained NLP classifier to the 430 recall records
# that Taxonomy v0.2 could not classify.
#
# IMPORTANT:
# These outputs are model-assisted suggestions.
# They are NOT verified ground-truth labels.
#
# Because LogisticRegression used class_weight="balanced",
# probability outputs should be treated as MODEL SCORES rather
# than perfectly calibrated probabilities.
# ============================================================


# ------------------------------------------------------------
# TRANSFORM RESIDUAL TEXT
# ------------------------------------------------------------

X_residual_nlp = hybrid_vectorizer.transform(
    nlp_residual_df["reason_clean"]
)


# ------------------------------------------------------------
# GENERATE CATEGORY SCORES
# ------------------------------------------------------------

residual_scores = nlp_classifier.predict_proba(
    X_residual_nlp
)


residual_score_df = pd.DataFrame(
    residual_scores,
    columns=[
        f"{category}_score"
        for category in risk_columns_v2
    ]
)


# ------------------------------------------------------------
# FIND HIGHEST-SCORING CATEGORY
# ------------------------------------------------------------

score_columns = residual_score_df.columns.tolist()


residual_score_df["top_model_score"] = (
    residual_score_df[score_columns]
    .max(axis=1)
)


residual_score_df["top_model_category"] = (
    residual_score_df[score_columns]
    .idxmax(axis=1)
    .str.replace(
        "_score",
        "",
        regex=False
    )
)


# ------------------------------------------------------------
# COMBINE WITH ORIGINAL RECALL DATA
# ------------------------------------------------------------

residual_model_review_df = pd.concat(
    [
        nlp_residual_df[
            [
                "recall_number",
                "reason_clean"
            ]
        ].reset_index(drop=True),

        residual_score_df
    ],
    axis=1
)


# ------------------------------------------------------------
# MODEL-SCORE BANDS
# ------------------------------------------------------------

residual_model_review_df["score_band"] = pd.cut(
    residual_model_review_df["top_model_score"],
    bins=[
        -0.01,
        0.50,
        0.70,
        0.85,
        1.00
    ],
    labels=[
        "<0.50",
        "0.50–0.70",
        "0.70–0.85",
        "0.85–1.00"
    ]
)


print("PHARMAOPT — RESIDUAL MODEL-ASSISTED CLASSIFICATION")
print("=" * 90)

print(
    "Residual recall records scored:",
    f"{len(residual_model_review_df):,}"
)

print(
    "\nTOP MODEL CATEGORY COUNTS"
)

print(
    residual_model_review_df[
        "top_model_category"
    ]
    .value_counts()
    .to_string()
)


print(
    "\nTOP MODEL SCORE DISTRIBUTION"
)

print(
    residual_model_review_df[
        "score_band"
    ]
    .value_counts(sort=False)
    .to_string()
)


print(
    "\nMODEL SCORE SUMMARY"
)

print(
    residual_model_review_df[
        "top_model_score"
    ]
    .describe()
    .round(4)
    .to_string()
)

PHARMAOPT — RESIDUAL MODEL-ASSISTED CLASSIFICATION
Residual recall records scored: 430

TOP MODEL CATEGORY COUNTS
top_model_category
manufacturing_gmp_risk               342
particulate_foreign_matter_risk       31
labeling_risk                         26
specification_failure_risk            13
physical_product_quality_risk          5
packaging_delivery_risk                5
contamination_risk                     3
regulatory_approval_risk               2
sterility_risk                         1
product_identity_formulation_risk      1
potency_dosage_risk                    1

TOP MODEL SCORE DISTRIBUTION
score_band
<0.50        369
0.50–0.70     19
0.70–0.85     11
0.85–1.00     31

MODEL SCORE SUMMARY
count    430.0000
mean       0.2730
std        0.2325
min        0.0983
25%        0.1556
50%        0.1556
75%        0.2775
max        1.0000


In [31]:
# ============================================================
# PHARMAOPT — RESIDUAL NLP HUMAN-REVIEW AUDIT
# ============================================================

print("PHARMAOPT — RESIDUAL NLP HUMAN-REVIEW AUDIT")
print("=" * 100)


review_examples = (
    residual_model_review_df
    .sort_values(
        "top_model_score",
        ascending=False
    )
)


for row in review_examples.head(25).itertuples():

    description = row.reason_clean

    if len(description) > 250:
        description = description[:250] + "..."

    print("\n" + "-" * 100)

    print(
        "Suggested category:",
        row.top_model_category
    )

    print(
        "Model score:",
        f"{row.top_model_score:.4f}"
    )

    print(
        "Recall:",
        row.recall_number
    )

    print(
        "Reason:",
        description
    )

PHARMAOPT — RESIDUAL NLP HUMAN-REVIEW AUDIT

----------------------------------------------------------------------------------------------------
Suggested category: sterility_risk
Model score: 1.0000
Recall: D-0243-2026
Reason: lack of assurance of sterlity

----------------------------------------------------------------------------------------------------
Suggested category: potency_dosage_risk
Model score: 0.9998
Recall: D-1172-2018
Reason: supterpotent drug

----------------------------------------------------------------------------------------------------
Suggested category: specification_failure_risk
Model score: 0.9977
Recall: D-1418-2015
Reason: falied ph specification

----------------------------------------------------------------------------------------------------
Suggested category: labeling_risk
Model score: 0.9942
Recall: D-0232-2022
Reason: labelling incorrect exp date

--------------------------------------------------------------------------------------------------

In [32]:
# ============================================================
# PHARMAOPT — UNIQUE RESIDUAL PREDICTION AUDIT
# ============================================================
#
# Multiple FDA recall records may contain identical reason text.
# We collapse identical normalized descriptions so repeated
# records do not make the NLP model appear more certain or
# more broadly applicable than it really is.
# ============================================================

unique_residual_predictions = (
    residual_model_review_df
    .groupby("reason_clean", as_index=False)
    .agg(
        record_count=("recall_number", "count"),
        top_model_category=("top_model_category", "first"),
        top_model_score=("top_model_score", "first")
    )
    .sort_values(
        "top_model_score",
        ascending=False
    )
    .reset_index(drop=True)
)


print("PHARMAOPT — UNIQUE RESIDUAL PREDICTION AUDIT")
print("=" * 90)

print(
    "Residual recall records:",
    f"{len(residual_model_review_df):,}"
)

print(
    "Unique residual descriptions:",
    f"{len(unique_residual_predictions):,}"
)


print("\nUNIQUE DESCRIPTIONS BY TOP-SCORE THRESHOLD")
print("=" * 90)


for threshold in [
    0.50,
    0.60,
    0.70,
    0.80,
    0.85,
    0.90,
    0.95
]:

    selected = unique_residual_predictions[
        unique_residual_predictions[
            "top_model_score"
        ] >= threshold
    ]

    print(
        f">= {threshold:.2f}: "
        f"{len(selected):>3} unique descriptions | "
        f"{selected['record_count'].sum():>3} recall records"
    )

PHARMAOPT — UNIQUE RESIDUAL PREDICTION AUDIT
Residual recall records: 430
Unique residual descriptions: 72

UNIQUE DESCRIPTIONS BY TOP-SCORE THRESHOLD
>= 0.50:  28 unique descriptions |  61 recall records
>= 0.60:  21 unique descriptions |  48 recall records
>= 0.70:  16 unique descriptions |  42 recall records
>= 0.80:  10 unique descriptions |  32 recall records
>= 0.85:   9 unique descriptions |  31 recall records
>= 0.90:   8 unique descriptions |  30 recall records
>= 0.95:   6 unique descriptions |  11 recall records


In [33]:
# ============================================================
# PHARMAOPT — HIGH-SCORE NLP CANDIDATE REVIEW
# ============================================================
#
# 0.85 is currently a REVIEW threshold, not an automatic
# classification threshold.
#
# No NLP-derived category becomes trusted simply because its
# model score exceeds this value.
# ============================================================

review_threshold = 0.85


high_score_candidates = (
    unique_residual_predictions[
        unique_residual_predictions[
            "top_model_score"
        ] >= review_threshold
    ]
    .copy()
)


print("PHARMAOPT — HIGH-SCORE NLP CANDIDATES")
print("=" * 100)

print(
    "Review threshold:",
    review_threshold
)

print(
    "Unique candidate descriptions:",
    len(high_score_candidates)
)

print(
    "Underlying recall records:",
    high_score_candidates[
        "record_count"
    ].sum()
)


print("\nCANDIDATE CATEGORY COUNTS")
print("=" * 100)

print(
    high_score_candidates[
        "top_model_category"
    ]
    .value_counts()
    .to_string()
)


print("\nCANDIDATE DESCRIPTIONS")
print("=" * 100)


for row in high_score_candidates.itertuples():

    description = row.reason_clean

    if len(description) > 300:
        description = (
            description[:300] + "..."
        )

    print("\n" + "-" * 100)

    print(
        f"Category: {row.top_model_category}"
    )

    print(
        f"Model score: {row.top_model_score:.4f}"
    )

    print(
        f"Recall records represented: {row.record_count}"
    )

    print(
        f"Reason: {description}"
    )

PHARMAOPT — HIGH-SCORE NLP CANDIDATES
Review threshold: 0.85
Unique candidate descriptions: 9
Underlying recall records: 31

CANDIDATE CATEGORY COUNTS
top_model_category
specification_failure_risk    3
manufacturing_gmp_risk        3
sterility_risk                1
potency_dosage_risk           1
labeling_risk                 1

CANDIDATE DESCRIPTIONS

----------------------------------------------------------------------------------------------------
Category: sterility_risk
Model score: 1.0000
Recall records represented: 1
Reason: lack of assurance of sterlity

----------------------------------------------------------------------------------------------------
Category: potency_dosage_risk
Model score: 0.9998
Recall records represented: 1
Reason: supterpotent drug

----------------------------------------------------------------------------------------------------
Category: specification_failure_risk
Model score: 0.9977
Recall records represented: 1
Reason: falied ph specification

-

In [34]:
# ============================================================
# PHARMAOPT — NLP REVIEW LAYER
# ============================================================
#
# High-score NLP suggestions are preserved separately from the
# original rule-based taxonomy.
#
# This prevents model-generated suggestions from silently
# becoming equivalent to deterministic taxonomy labels.
# ============================================================

nlp_review_threshold = 0.85


nlp_review_candidates = (
    unique_residual_predictions[
        unique_residual_predictions[
            "top_model_score"
        ] >= nlp_review_threshold
    ]
    .copy()
)


nlp_review_candidates["review_status"] = (
    "candidate_for_human_validation"
)


nlp_review_candidates["signal_source"] = (
    "nlp_model_v0.1"
)


nlp_review_candidates = (
    nlp_review_candidates[
        [
            "reason_clean",
            "record_count",
            "top_model_category",
            "top_model_score",
            "review_status",
            "signal_source"
        ]
    ]
    .sort_values(
        "top_model_score",
        ascending=False
    )
    .reset_index(drop=True)
)


print("PHARMAOPT — NLP REVIEW LAYER")
print("=" * 100)

print(
    "Unique NLP candidates:",
    len(nlp_review_candidates)
)

print(
    "Recall records represented:",
    nlp_review_candidates[
        "record_count"
    ].sum()
)

print(
    "Residual descriptions remaining unresolved:",
    len(unique_residual_predictions)
    - len(nlp_review_candidates)
)

print(
    "\nCandidate table:"
)

display(nlp_review_candidates)

PHARMAOPT — NLP REVIEW LAYER
Unique NLP candidates: 9
Recall records represented: 31
Residual descriptions remaining unresolved: 63

Candidate table:


,reason_clean,record_count,top_model_category,top_model_score,review_status,signal_source
0,lack of assurance of sterlity,1,sterility_risk,0.999954,candidate_for_human_validation,nlp_model_v0.1
1,supterpotent drug,1,potency_dosage_risk,0.999781,candidate_for_human_validation,nlp_model_v0.1
2,falied ph specification,1,specification_failure_risk,0.997709,candidate_for_human_validation,nlp_model_v0.1
3,labelling incorrect exp date,1,labeling_risk,0.994223,candidate_for_human_validation,nlp_model_v0.1
4,finished product specification did not meet us...,1,specification_failure_risk,0.993915,candidate_for_human_validation,nlp_model_v0.1
5,ggmp deviations possible contamination with im...,6,manufacturing_gmp_risk,0.955096,candidate_for_human_validation,nlp_model_v0.1
6,failed dissolution testing failed 24 month dis...,1,specification_failure_risk,0.938730,candidate_for_human_validation,nlp_model_v0.1
7,gcmp deviations fda analysis confirmed presenc...,18,manufacturing_gmp_risk,0.922425,candidate_for_human_validation,nlp_model_v0.1
8,cmgp deviations presence of ndma impurity dete...,1,manufacturing_gmp_risk,0.889831,candidate_for_human_validation,nlp_model_v0.1


In [35]:
# ============================================================
# PHARMAOPT — QUALITATIVE INTELLIGENCE COVERAGE SUMMARY
# ============================================================

total_recall_records = len(recall_text_df)

rule_classified_records = int(
    (
        recall_text_df[
            "unclassified_risk_v2"
        ] == 0
    ).sum()
)

rule_unclassified_records = int(
    (
        recall_text_df[
            "unclassified_risk_v2"
        ] == 1
    ).sum()
)

nlp_candidate_records = int(
    nlp_review_candidates[
        "record_count"
    ].sum()
)

still_unresolved_records = (
    rule_unclassified_records
    - nlp_candidate_records
)


rule_coverage = (
    rule_classified_records
    / total_recall_records
)

nlp_incremental_coverage = (
    nlp_candidate_records
    / total_recall_records
)

potential_combined_coverage = (
    rule_classified_records
    + nlp_candidate_records
) / total_recall_records


print(
    "PHARMAOPT — QUALITATIVE INTELLIGENCE COVERAGE"
)
print("=" * 80)

print(
    "Total recall records:",
    f"{total_recall_records:,}"
)

print(
    "Rule-classified records:",
    f"{rule_classified_records:,}"
)

print(
    "Rule coverage:",
    f"{rule_coverage:.2%}"
)

print(
    "\nRule-unclassified records:",
    f"{rule_unclassified_records:,}"
)

print(
    "High-score NLP review candidates:",
    f"{nlp_candidate_records:,}"
)

print(
    "Still unresolved:",
    f"{still_unresolved_records:,}"
)

print(
    "\nIncremental records surfaced by NLP:",
    f"{nlp_incremental_coverage:.2%}"
)

print(
    "Potential combined coverage after review:",
    f"{potential_combined_coverage:.2%}"
)

PHARMAOPT — QUALITATIVE INTELLIGENCE COVERAGE
Total recall records: 17,975
Rule-classified records: 17,545
Rule coverage: 97.61%

Rule-unclassified records: 430
High-score NLP review candidates: 31
Still unresolved: 399

Incremental records surfaced by NLP: 0.17%
Potential combined coverage after review: 97.78%


In [36]:
# PharmaOpt Qualitative Supply-Risk Intelligence — v0.1 Conclusion

Notebook 09 developed an auditable qualitative risk-intelligence layer from FDA recall text.

## Rule-Based Taxonomy

An interpretable multi-label taxonomy was developed for operational risk themes including sterility, manufacturing/GMP, contamination, particulate matter, potency, specification failure, stability, packaging, labeling, regulatory approval, storage conditions, product identity/formulation, and physical product quality.

Taxonomy v0.2 classified approximately 97.6% of FDA recall records while preserving unmatched text rather than forcing every recall into a category.

## Residual NLP Analysis

The remaining 430 recall records represented only 72 unique normalized descriptions.

TF-IDF + K-means clustering was tested as an exploratory unsupervised NLP method. The best tested silhouette score was 0.0747 at k=10, indicating weak overall cluster separation. Therefore, cluster membership was not adopted as an operational risk feature.

## Weakly Supervised NLP

The rule-derived taxonomy was then used as weak supervision for a multi-label text classifier using word-level and character-level TF-IDF features with logistic regression.

To reduce leakage, identical normalized recall descriptions were assigned entirely to either training or holdout data.

Holdout performance:

- Micro Precision: 0.9939
- Micro Recall: 0.9755
- Micro F1: 0.9846
- Macro F1: 0.9432
- Hamming Loss: 0.0027

These metrics measure reproduction and generalization of the rule-derived taxonomy, not independent real-world recall-cause accuracy.

## Residual Model Assistance

The NLP model was applied to the residual text only as a review-assistance layer.

At a conservative model-score review threshold of 0.85, the model surfaced nine unique residual descriptions representing 31 recall records. Several represented spelling or wording variants of established risk concepts, including sterility, potency, specification failure, labeling, and GMP deviations.

Most residual records remained below the review threshold and were intentionally left unresolved.

## Methodological Decision

PharmaOpt does not force all qualitative text into a risk category.

The qualitative architecture therefore uses:

1. interpretable deterministic rules for known patterns;
2. NLP-assisted suggestions for strong unresolved language patterns;
3. explicit human review for model-assisted signals; and
4. an unresolved state when evidence is insufficient.

NLP-derived signals are decision-support features and should not be interpreted as verified causal explanations of shortages or recalls.

## Next Phase

The next modeling phase will convert historical qualitative recall-risk signals into temporally valid package-level features and test whether they improve PharmaOpt's existing shortage-risk prediction model beyond FDA quantitative features and CMS Part D utilization signals.

SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (3253725679.py, line 3)